# Specialiserede Modeller — 1: Beslutningstræer & random forests

Neurale netværk er ML-verdenens schweizerkniv. Men en schweizerkniv
er sjældent det BEDSTE værktøj til noget som helst. I dette emne møder du modeller, der
er specialiserede — og vi starter med den model, der regerer på **tabeldata**:
**beslutningstræet** og dets storebror, **random forest**.

Dagens data: 8.124 svampe. Spørgsmålet: **tør du spise den?**

> Denne notebook er selvkørende — du kan tage emnets notebooks i den rækkefølge, du vil. Der er med vilje flere opgaver, end du kan nå. Opgaver mærket **(find fejlen)** indeholder en bevidst fejl, som skal findes og rettes. Nederst ligger et par **ekstra opgaver**, hvis du får lyst til mere.
>
> Noget af det her er nyt og kan føles udfordrende i starten — og det er helt okay. Vi forklarer hvert skridt så klart og tydeligt, vi kan, og der er et hint til hver opgave, hvis du går i stå. Tag dig endelig god tid.

## Setup

In [ ]:
# Henter svampe-data fra GitHub (Plan B: upload mushrooms.csv via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/mushrooms.csv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

df = pd.read_csv("mushrooms.csv")
df.head()

# 1: Beslutningstræet — modellen der stiller spørgsmål

Kender I spillet *20 spørgsmål* (eller Akinator)? Man gætter hvad som helst ved at stille
ja/nej-spørgsmål, hvor hvert svar skærer halvdelen af mulighederne væk. Et
**beslutningstræ** er præcis dét — nu lært automatisk fra data:

> *Lugter svampen af noget?* → ja → *Er lugten mandel-agtig?* → nej → **GIFTIG**

Træet er på mange måder det modsatte af et neuralt netværk: det kan ikke det samme —
men det kan noget, intet netværk kan: **man kan læse det**.

## Dataene: 8.124 svampe

Hver række er en svamp, og `class` fortæller om den er **e**dible (spiselig) eller
**p**oisonous (giftig). Kig på tabellen ovenfor: ALLE kolonner er bogstavkoder —
kategorier, ikke tal! Fx `odor` (lugt): `a`=mandel, `l`=anis, `n`=ingen lugt,
`f`=modbydelig, `p`=skarp... og `cap-color` (hattens farve), `habitat` (voksested) osv.

In [ ]:
print(df.shape)
print(df["class"].value_counts())     # e = spiselig, p = giftig
print()
print(df["odor"].value_counts())      # lugt: n=ingen, f=modbydelig, a=mandel, l=anis, ...

## Tekst → tal: `pd.get_dummies`

Modeller spiser kun tal (Intro-ML-lektion nr. 1!). Men `odor` er ikke et tal, og det
ville være løgn at kode mandel=1, anis=2, ingen=3 — anis er jo ikke "dobbelt så meget"
som mandel! Løsningen er **one-hot encoding**: hver kategori får sin egen 0/1-kolonne
(`odor_a`, `odor_l`, `odor_n`,...). Det klarer `pd.get_dummies` i én linje:

In [ ]:
X = pd.get_dummies(df.drop(columns=["class"]))     # alle features, one-hot-kodet
y = (df["class"] == "p").astype(int)               # 1 = giftig, 0 = spiselig

print("før: ", df.shape, "— 22 tekstkolonner")
print("efter:", X.shape, "— kun 0/1-kolonner")
X.head(3)

## sklearn — ét mønster, hundrede modeller

Træer bygger vi ikke i PyTorch men i **scikit-learn** (sklearn) — bibliotekét for
"klassisk" ML. Og her er dagens vigtigste generelle færdighed: i sklearn ligner ALLE
modeller hinanden:

```python
model = EnEllerAndenModel()      # 1. opret
model.fit(X_traen, y_traen)      # 2. træn (én linje — intet træningsloop!)
model.predict(X_test)            # 3. forudsig
model.score(X_test, y_test)      # 4. mål accuracy
```

Lærer I det mønster, kan I bruge hundredvis af modeller. Ingen `zero_grad`, ingen
epoker — sklearn klarer alt indeni. Vi splitter i train/test, og starter
med et bevidst LILLE træ (`max_depth=2` — højst 2 spørgsmål dybt):

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

trae = DecisionTreeClassifier(max_depth=2, random_state=42)
trae.fit(X_train, y_train)

print(f"accuracy på træningsdata: {trae.score(X_train, y_train):.1%}")
print(f"accuracy på testdata:     {trae.score(X_test, y_test):.1%}")

94 % med højst to spørgsmål! Og nu det magiske — lad os **læse** modellen:

In [ ]:
plt.figure(figsize=(13, 6))
plot_tree(trae,
          feature_names=X.columns,
          class_names=["spiselig", "giftig"],
          filled=True, fontsize=10)
plt.show()

**Sådan læses en kasse:** øverst står spørgsmålet (fx `odor_n <= 0.5` betyder "er
`odor_n` 0? — altså: lugter svampen af NOGET?"). `samples` er antal svampe i kassen,
`value` er [spiselige, giftige], og farven viser flertallet (jo mørkere, jo mere
"ren" er kassen — dvs. jo mere enig er svampene i kassen om svaret).

Træet har selv fundet ud af, at **lugt** er det bedste spørgsmål — præcis som en rigtig
svampeguide! At vælge det spørgsmål, der gør kasserne mest rene, er hele
træ-algoritmen. (Renhed måles med *gini* eller *entropi* — samme idé som når man i
20 spørgsmål vælger det spørgsmål, der skærer flest muligheder væk.)

## Hvor dybt skal træet være?

`max_depth` er træets vigtigste indstilling: for lavt = for dumt, for dybt = risiko for
udenadslære (overfitting — fælden fra før). Og hvilke spørgsmål gjorde mest nytte?
Det fortæller `feature_importances_`:

In [ ]:
trae4 = DecisionTreeClassifier(max_depth=4, random_state=42)
trae4.fit(X_train, y_train)

importance = pd.Series(trae4.feature_importances_, index=X.columns)
importance.sort_values().tail(8).plot(kind="barh")
plt.title("Hvilke spørgsmål gør mest nytte?")
plt.xlabel("vigtighed")
plt.show()

### Opgaver

##### Opgave 1.1
Vi følger en enkelt svamp hele vejen gennem træet — i hånden, uden kode.

Kig på træet fra `plot_tree`-cellen lige ovenfor. Vores svamp lugter af ingenting, så `odor_n = 1`.

Prøv at aflæse: hvilken vej sendes svampen ved det allerførste spørgsmål, og hvad ender træet med at gætte? Er du enig — tør du selv spise den?

Husk: spørgsmålet øverst er `odor_n <= 0.5`. Det er kun sandt, når `odor_n` er 0 — og vores svamp har `odor_n = 1`, så hvilken gren fører "falsk" ned ad?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 1.2
Vi undersøger, hvad træets dybde betyder for, hvor godt det rammer.

I cellen nedenfor kan du ændre `max_depth`. Prøv efter tur værdierne 1, 3, 5 og None (None betyder ubegrænset dybde), og notér både train- og test-accuracy hver gang.

Se om du kan svare: hvornår rammer træet 100 %, og bliver test-accuracy nogensinde DÅRLIGERE af mere dybde her?

Husk: før tallene ind i et lille skema — så er det tydeligt at se, hvornår de to tal begynder at glide fra hinanden.

In [ ]:
trae = DecisionTreeClassifier(max_depth=1, random_state=42)   # ← prøv 1, 3, 5, None
trae.fit(X_train, y_train)
print(f"train: {trae.score(X_train, y_train):.1%}   test: {trae.score(X_test, y_test):.1%}")

##### Opgave 1.3
Nedenfor måler vi et træ med dybde 3 på både trænings- og testdata.

`score(X, y)` tager to ting: hvilke svampe den skal se på (`X`), og hvad det rigtige svar er (`y`). Vi har allerede fyldt `X`-delen ud på hver linje — prøv at udfylde det rigtige facit til hver.

Husk: `X_train` hører sammen med `y_train`, og `X_test` hører sammen med `y_test` — som par, der skal matche.

In [ ]:
trae = DecisionTreeClassifier(max_depth=3, random_state=42)
trae.fit(X_train, y_train)
print("train:", trae.score(X_train, ...))   # <-- udfyld: de rigtige svar for træningssættet
print("test: ", trae.score(X_test, ...))    # <-- udfyld: de rigtige svar for testsættet

##### Opgave 1.4 (find fejlen)
En kammerat praler: *"Min model rammer 100 % — og jeg behøvede ikke engang træne på træningsdataene!"*

Kig godt på, hvilke data modellen bliver trænet på, og hvilke den bliver målt på. Prøv at forklare, hvad der er galt — og hvorfor tallet er værdiløst, selvom det er ægte nok.

Husk: hvad står der i `fit(...)`, og hvad står der i `score(...)`? Er det de samme svampe? Tænk på eksamensreglen fra Intro-ML — man må ikke få facit med til eksamen.

In [ ]:
genius_model = DecisionTreeClassifier(random_state=42)
genius_model.fit(X_test, y_test)
print(f"accuracy: {genius_model.score(X_test, y_test):.1%} — GENIALT?!")

##### Opgave 1.5
Vi ser på, hvilket spørgsmål træet selv synes er vigtigst.

Kør feature-importance-cellen ovenfor igen, og kig på den øverste bjælke. Prøv at aflæse, hvilken kolonne der vinder suverænt — og hvad den kolonne betyder oversat til dansk svampejæger-sprog.

Husk: kolonnenavnene har formen `feature_kategori`, fx `odor_n` = "lugt: ingen". Den vindende kolonne fortæller altså noget om ét bestemt kendetegn.

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 1.6
Træet uden dybdegrænse rammer **100 % på testdata**. Perfekte tal er normalt noget, man skal være mistænksom over for.

Prøv at tage stilling til tre ting: (a) hvorfor er 100 % normalt et alarmsignal, (b) hvad gør, at tallet faktisk er ægte her, og (c) ville du spise en svamp, alene fordi modellen siger "spiselig"?

Husk: en klassisk fælde er overfitting — modellen lærte støjen udenad. Findes der overhovedet støj i de her svampedata, eller er reglerne knivskarpe?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 1.7
Vi tager træets stærkeste kort fra det: lugten. Cellen nedenfor fjerner hele lugte-kolonnen (før `get_dummies`) og træner et nyt træ.

Prøv at skrue `max_depth` op og se, hvor god test-accuracy kan blive uden lugt — og hvor dyb modellen skal være for at nå derop.

Husk: i opgave 1.5 så du, at lugt var det suverænt vigtigste spørgsmål. Uden den bliver træet nødt til at stille mange flere små spørgsmål for at nå samme sikkerhed.

In [ ]:
df_uden_lugt = df.drop(columns=["odor"])
X2 = pd.get_dummies(df_uden_lugt.drop(columns=["class"]))
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.2, random_state=42)

trae = DecisionTreeClassifier(max_depth=2, random_state=42)   # ← prøv flere dybder
trae.fit(X2_train, y2_train)
print(f"uden lugt: test-accuracy {trae.score(X2_test, y2_test):.1%}")

##### Opgave 1.8
Nedenfor lader vi træet gætte og holder gættene op mod facit.

`predict` tager de svampe, modellen skal gætte på. Prøv at udfylde, hvilke svampe det skal være — vi vil gerne se gættet for testsvampene.

Husk: modellen er trænet på `X_train`, men vi vil teste den på svampe, den ikke har set før. Hvilket X passer til det?

In [ ]:
trae = DecisionTreeClassifier(max_depth=4, random_state=42)
trae.fit(X_train, y_train)

pred = trae.predict(...)   # <-- udfyld: test-svampene modellen skal gætte på
print("modellens gæt:", pred[:5])
print("facit:        ", y_test.values[:5])

##### Opgave 1.9
Vi prøver træets anden måde at måle "renhed" på.

Et træ kan bruge `criterion="gini"` (standard) eller `criterion="entropy"` (målet fra 20 spørgsmål). Prøv at skifte mellem de to ved et par forskellige dybder og se, om det gør nogen forskel her.

Husk: begge mål belønner det samme — spørgsmål, der gør kasserne så rene som muligt. Forvent derfor sjældent stor forskel; pointen er, at du ved, knappen findes.

In [ ]:
trae = DecisionTreeClassifier(max_depth=3, criterion="gini", random_state=42)   # ← prøv "entropy"
trae.fit(X_train, y_train)
print(f"test: {trae.score(X_test, y_test):.1%}")

# 2: Random forest — visdom fra en flok

Ét træ har en akilleshæl: det tager sine beslutninger MEGET bogstaveligt. Er der fejl
eller støj i træningsdataene, lærer et dybt træ støjen udenad — klassisk overfitting.

Løsningen er lige så enkel som genial: **træn 100 forskellige træer og lad dem
stemme**. Hvert træ ser et tilfældigt udsnit af data og et tilfældigt udvalg af
features (derfor "random"), så de laver FORSKELLIGE fejl — og flertalsafstemningen
visker fejlene ud. Som at spørge 100 svampejægere i stedet for én excentrisk ekspert.

I sklearn er det bogstaveligt talt en anden klasse — samme fit/predict/score:

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

start = time.time()
skov = RandomForestClassifier(n_estimators=100, random_state=42)   # 100 træer
skov.fit(X_train, y_train)
print(f"test-accuracy: {skov.score(X_test, y_test):.1%}   (trænet på {time.time() - start:.2f} s)")

## Hvor skoven for alvor skinner: støjede data

Svampedataene er FOR pæne til at vise forskellen. Så lad os sabotere dem: vi "flipper"
10 % af træningslabels (som om nogen har tastet forkert i felterne) og ser, hvem der
holder hovedet koldt — ét dybt træ eller skoven:

In [ ]:
rng = np.random.default_rng(42)

y_noise = y_train.values.copy()
flip = rng.choice(len(y_noise), size=int(0.10 * len(y_noise)), replace=False)
y_noise[flip] = 1 - y_noise[flip]            # 10 % af svarene er nu FORKERTE

trae = DecisionTreeClassifier(random_state=42)
trae.fit(X_train, y_noise)
skov = RandomForestClassifier(n_estimators=100, random_state=42)
skov.fit(X_train, y_noise)

print("(målt på RENE testdata)")
print(f"ét dybt træ: {trae.score(X_test, y_test):.1%}")
print(f"skov:        {skov.score(X_test, y_test):.1%}")

Se DEN forskel! Træet lærte tastefejlene udenad og faldt til ~86 % — skoven stemte
sig igennem støjen og holder ~100 %. Det er derfor, random forests i den virkelige
verden (hvor data ALTID er støjede) er et af de mest brugte værktøjer overhovedet.

## Duellen: skov mod neuralt netværk

Sidste spørgsmål: hvad med det neurale netværk? Lad os lade dem
kæmpe om svampene på fair vilkår — samme data, samme test:

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

X_train_t = torch.tensor(X_train.values.astype("float32"))
X_test_t = torch.tensor(X_test.values.astype("float32"))
y_train_t = torch.tensor(y_train.values.astype("float32"))
y_test_t = torch.tensor(y_test.values.astype("float32"))

network = nn.Sequential(nn.Linear(X.shape[1], 32), nn.ReLU(),
                         nn.Linear(32, 1), nn.Sigmoid())
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(network.parameters(), lr=0.01)

start = time.time()
for epoch in range(200):                       # de fem trin
    optimizer.zero_grad()
    loss = loss_fn(network(X_train_t).squeeze(), y_train_t)
    loss.backward()
    optimizer.step()
nn_tid = time.time() - start

with torch.no_grad():
    nn_pred = (network(X_test_t).squeeze() > 0.5).float()
print(f"neuralt netværk: {(nn_pred == y_test_t).float().mean().item():.1%} på {nn_tid:.1f} s")

start = time.time()
skov = RandomForestClassifier(n_estimators=100, random_state=42)
skov.fit(X_train, y_train)
print(f"random forest:   {skov.score(X_test, y_test):.1%} på {time.time() - start:.2f} s")

Begge rammer (næsten) perfekt — men skoven er mange gange hurtigere, krævede NUL
hyperparameter-fifleri (ingen læringsrate, ingen epoker, ingen standardisering!), og
dens enkelt-træer kan læses. På tabeldata er træ-modeller derfor næsten altid det
første, man prøver. Netværkenes hjemmebane er billeder, lyd og sprog — det ser I i de
andre notebooks. **Det rigtige værktøj til det rigtige job.**

### Opgaver

##### Opgave 2.1
Vi undersøger, hvor mange træer en skov egentlig har brug for.

Cellen nedenfor træner på de støjede labels (`y_noise`), hvor forskellen mellem få og mange træer kan ses. Prøv `n_estimators` på 1, 5, 25 og 100 efter tur og se, hvornår forbedringen flader ud.

Husk: ét træ alene lærer let støjen udenad. Jo flere træer der stemmer, jo mere visker de hinandens tilfældige fejl ud — men på et tidspunkt er der ikke mere at hente.

In [ ]:
skov = RandomForestClassifier(n_estimators=1, random_state=42)   # ← prøv 1, 5, 25, 100
skov.fit(X_train, y_noise)
print(f"test: {skov.score(X_test, y_test):.1%}")

##### Opgave 2.2
Vi skruer op for sabotagen og ser, hvor robust skoven er.

I cellen nedenfor kan du ændre `fraction` — andelen af træningssvar, der vendes om. Prøv 0.10, 0.20 og 0.30 og sammenlign træ mod skov hver gang.

Prøv at svare: hvem knækker først, og hvor meget støj skal der til, før skoven for alvor begynder at lide?

Husk: skovens styrke er, at 100 træer laver FORSKELLIGE fejl. Jo mere støj, jo sværere bliver det dog for flertallet at pege på det rigtige svar.

In [ ]:
fraction = 0.10   # ← prøv 0.10, 0.20, 0.30
y_noise2 = y_train.values.copy()
flip = rng.choice(len(y_noise2), size=int(fraction * len(y_noise2)), replace=False)
y_noise2[flip] = 1 - y_noise2[flip]

trae = DecisionTreeClassifier(random_state=42)
trae.fit(X_train, y_noise2)
skov = RandomForestClassifier(n_estimators=100, random_state=42)
skov.fit(X_train, y_noise2)
print(f"støj {fraction:.0%}: træ {trae.score(X_test, y_test):.1%} | skov {skov.score(X_test, y_test):.1%}")

##### Opgave 2.3
Vi laver samme vigtigheds-plot som i afsnit 1, men nu for hele SKOVEN. En skov har også `feature_importances_` — det er gennemsnittet over alle 100 træer.

Vi har allerede sat `index=X.columns`, så bjælkerne får de rigtige navne. Prøv at udfylde selve vigtighedstallene.

Sammenlign bagefter med det enkelte træs plot fra afsnit 1: er vigtigheden mere spredt ud nu?

Husk mønstret fra afsnit 1: der stod `pd.Series(trae4.feature_importances_, index=...)`. Skoven `skov` har præcis den samme egenskab.

In [ ]:
skov = RandomForestClassifier(n_estimators=100, random_state=42)
skov.fit(X_train, y_train)

importance = pd.Series(..., index=X.columns)   # <-- udfyld: skovens vigtighedstal (den har feature_importances_ ligesom træet)
importance.sort_values().tail(10).plot(kind="barh")
plt.xlabel("vigtighed")
plt.show()

##### Opgave 2.4 (find fejlen)
En kammerat springer over, hvor gærdet er lavest, og fodrer skoven med den RÅ tabel — helt uden `get_dummies`. Cellen crasher.

Prøv at køre den, læse fejlbeskeden og rette koden, så skoven får one-hot-kodede tal i stedet. Hvad er det, sklearn prøver at fortælle os?

Husk: modeller spiser kun tal — den allerførste ML-lektion. Hvor i notebooken lavede vi bogstavkoderne om til 0/1-kolonner?

In [ ]:
skov = RandomForestClassifier(n_estimators=100, random_state=42)
skov.fit(df.drop(columns=["class"]), y)
print(skov.score(df.drop(columns=["class"]), y))

##### Opgave 2.5
Vi ser på prisen for skovens robusthed.

Dybde-2-træet kunne vi læse direkte i `plot_tree`. Prøv at forestille dig en skov på 100 træer, der hver er 15 spørgsmål dybe — kan man stadig læse den?

Overvej: hvad har vi ofret for robustheden, og hvornår kan det offer være et problem (tænk fx på en bank, der afviser dit lån og skal kunne forklare hvorfor)?

Husk: ét træ er en læsbar kæde af ja/nej-spørgsmål. 100 dybe træer, der stemmer, giver et godt svar — men ikke længere en forklaring, du kan følge med fingeren.

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 2.6
Vi samler duellen mellem skov og netværk i ét overblik.

Kør NN-duellen ovenfor (hvis du ikke allerede har gjort det). Prøv så at udfylde sammenligningstabellen med dine egne ord — og kår en vinder *for netop dette datasæt*:

| | Random forest | Neuralt netværk |
|---|---|---|
| Accuracy | $\dots$ | $\dots$ |
| Træningstid | $\dots$ | $\dots$ |
| Krævede fifleri (lr, epoker, standardisering...)? | $\dots$ | $\dots$ |
| Kan modellen læses? | $\dots$ | $\dots$ |

Husk: alle fire tal og svar står allerede i output fra cellen ovenfor og i afsnittets tekst — du skal kun aflæse dem og skrive dem ind i skemaet.

## Ekstra opgaver

Her er nogle ekstra udfordringer, hvis du er nået hele vejen igennem og har lyst til mere. De bygger videre på det, du allerede har lavet, og du kan tage dem i den rækkefølge, du vil.

##### Ekstra 1
Vi tegner nu hele historien om dybde på én graf: en for-løkke træner et træ for hver dybde fra 1 til 10 og gemmer både train- og test-accuracy.

Prøv at udfylde de to linjer, der lægger accuracy ind i listerne. Kig bagefter: hvor flader kurverne ud, og åbner der sig et gab mellem dem?

Husk: du målte allerede accuracy i opgave 1.3 med `trae.score(X, y)` — det er præcis samme kald, du skal bruge her, nu for hhv. train og test.

In [ ]:
depths = range(1, 11)
train_acc = []
test_acc = []
for depth in depths:
    trae = DecisionTreeClassifier(max_depth=depth, random_state=42)
    trae.fit(X_train, y_train)
    train_acc.append(...)   # <-- udfyld: træets accuracy på træningsdata
    test_acc.append(...)    # <-- udfyld: træets accuracy på testdata

plt.plot(depths, train_acc, "o-", label="train")
plt.plot(depths, test_acc, "o-", label="test")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.legend()
plt.show()

##### Ekstra 2
Vi presser skoven på mængden af træningsdata: hvor LIDT kan den nøjes med?

Cellen nedenfor træner på tilfældige udsnit på 50, 200, 1000 og alle rækker og plotter test-accuracy mod træningsstørrelse. Prøv at køre den og se, hvor lidt data der egentlig skal til her.

Husk: x-aksen er logaritmisk (`plt.xscale("log")`), så de små udsnit får god plads. Kig efter, hvor tidligt kurven flader ud.

In [ ]:
sizes = [50, 200, 1000, len(X_train)]
results = []
for n in sizes:
    slice = X_train.sample(n=n, random_state=42)
    skov = RandomForestClassifier(n_estimators=100, random_state=42)
    skov.fit(slice, y_train.loc[slice.index])
    results.append(skov.score(X_test, y_test))
    print(f"{n:5d} rækker: {results[-1]:.1%}")

plt.plot(sizes, results, "o-")
plt.xscale("log")
plt.xlabel("antal træningsrækker")
plt.ylabel("test-accuracy")
plt.show()

##### Ekstra 3
Du starter i praktik hos et firma, der giver dig et nyt TABELDATASÆT (kunder, salg, sensorer — hvad som helst) og spørger: "kan du forudsige X?".

Prøv at tage stilling: hvilken model griber du til FØRST, og hvad er din begrundelse? Og hvornår ville du i stedet vælge et neuralt netværk?

Husk: hele afsnittets pointe var "det rigtige værktøj til det rigtige job". På ryddelig tabeldata plejer træ-modeller at være det hurtige førstevalg; netværkenes hjemmebane er billeder, lyd og sprog.

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*